# Cuantificación de mercado RCD — Regemac S.A. (§1.1b-ii)
### Reproducción completa de los cálculos del entregable "Dimensión y participación de mercado de Regemac en la gestión de RCD (Región Metropolitana)"

**Fuentes de datos:**
1. `Guias_202101-202608_con_info_de_app.xlsx` — guías de despacho de contenedores (hoja `Calculos`), entregado por Regemac. Corresponde a **contenedores con RCD mezclado que llegan al pozo**, no a material ya segregado/valorizado.
2. `Reporte envíos [Mes].csv` (enero a agosto 2026, 8 archivos) — despachos granulares de 2026, con el tamaño real de cada contenedor (`Tipo contenedor`). Se usan para *validar* el pivote interno de (1).
3. `Materiales_reciclados.xlsx` (hoja `Total`) — registro de ventas de material ya segregado (fierro, madera, aluminio, cartón, plástico, neumáticos), entregado por Regemac.
4. RETC/SINADER 2024, Ministerio del Medio Ambiente — declaraciones de movimiento de residuos, capítulo LER 17 (RCD). Se descarga directo desde el repositorio público que usa el equipo (`fernando25132/Evalpro`), que es una réplica del dataset oficial `datosretc.mma.gob.cl` ya usado en `bases_de_datos_residuos_2024_ampliado.ipynb`.
5. CChC/CDT, Informe MACh (dic-2025) — tasa de crecimiento de la inversión en construcción, usada como proxy de crecimiento del mercado de RCD 2025-2026 (no hay proyección sectorial directa).

**Etiquetas de procedencia**: `[CLIENT DATA]` dato entregado por Regemac · `[PUBLIC SOURCE]` fuente pública citada · `[OWN ESTIMATE]` estimación propia con método explícito · `[ASSUMPTION]` supuesto declarado, con rango y candidatura a sensibilidad.


## 0. Configuración y carga de archivos

Este cuaderno busca los archivos de Regemac en una carpeta `data/` junto al notebook (o en el mismo directorio). Si no los encuentra y detecta que está corriendo en Google Colab, ofrece subirlos manualmente. Los nombres pueden venir con o sin tildes/mayúsculas — la búsqueda es flexible.

In [ ]:
import pandas as pd
import numpy as np
import glob, os, re, unicodedata, io

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def _norm(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())

SEARCH_DIRS = ["data", ".", "./drive/MyDrive/Evalpro"]

def find_file(*keywords):
    """Busca, entre SEARCH_DIRS, un archivo cuyo nombre contenga todas las keywords
    (normalizadas: sin tildes, sin espacios, minúsculas)."""
    keys = [_norm(k) for k in keywords]
    candidates = []
    for d in SEARCH_DIRS:
        if not os.path.isdir(d):
            continue
        for path in glob.glob(os.path.join(d, "*")):
            name_norm = _norm(os.path.basename(path))
            if all(k in name_norm for k in keys):
                candidates.append(path)
    if candidates:
        return sorted(candidates)[0]
    # Intento en Colab: pedir upload si no se encontró
    try:
        from google.colab import files  # noqa
        print(f"No encontré un archivo con las palabras clave {keywords}. Súbelo ahora:")
        uploaded = files.upload()
        for name in uploaded:
            if all(k in _norm(name) for k in keys):
                return name
    except ImportError:
        pass
    raise FileNotFoundError(
        f"No encontré ningún archivo con las palabras clave {keywords} en {SEARCH_DIRS}. "
        f"Colócalo en una carpeta 'data/' junto a este notebook."
    )

PATH_GUIAS = find_file("guias", "202101")
PATH_MATERIALES = find_file("materiales", "reciclados")
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto"]
# El normalizador ya quita tildes/mayúsculas, así que basta con buscar por el nombre del mes
# (todos los "Reporte envíos [Mes].csv" son el único archivo de cada carpeta que contiene ese mes).
PATHS_ENVIOS = {m: find_file(m) for m in MESES}

print("Guías:", PATH_GUIAS)
print("Materiales reciclados:", PATH_MATERIALES)
for m, p in PATHS_ENVIOS.items():
    print(f"Reporte envíos {m}:", p)


Guías: data/Guias_202101-202608_con_info_de_app.xlsx
Materiales reciclados: data/Materiales_reciclados.xlsx
Reporte envíos Enero: data/Reporte envios Enero.csv
Reporte envíos Febrero: data/Reporte envios Febrero.csv
Reporte envíos Marzo: data/Reporte envios Marzo.csv
Reporte envíos Abril: data/Reporte envios Abril.csv
Reporte envíos Mayo: data/Reporte envios Mayo.csv
Reporte envíos Junio: data/Reporte envios Junio.csv
Reporte envíos Julio: data/Reporte envios Julio.csv
Reporte envíos Agosto: data/Reporte envios Agosto.csv


## 1. Tamaño del mercado RCD en la Región Metropolitana — RETC/SINADER 2024 `[PUBLIC SOURCE]`

Se descarga el mismo dataset (réplica pública) que usa `bases_de_datos_residuos_2024_ampliado.ipynb` del equipo, y se aplican los mismos dos filtros: capítulo LER 17 (RCD) y `region == 'Metropolitana de Santiago'`.


In [ ]:
RETC_URLS = [
    "https://raw.githubusercontent.com/fernando25132/Evalpro/refs/heads/main/df-sinader-2024-ckan_Datos_Parte1.csv",
    "https://raw.githubusercontent.com/fernando25132/Evalpro/refs/heads/main/df-sinader-2024-ckan_Datos_Parte2.csv",
]

df_retc = pd.concat([pd.read_csv(u, low_memory=False) for u in RETC_URLS], ignore_index=True)
print(f"Filas totales RETC/SINADER 2024 (todos los residuos, todo Chile): {len(df_retc):,}")

df_rcd = df_retc[df_retc["LER_numero_capitulo"] == 17].copy()
df_rcd_rm = df_rcd[df_rcd["region"] == "Metropolitana de Santiago"].copy()

print(f"Declaraciones de RCD (LER cap. 17) a nivel nacional: {len(df_rcd):,}")
print(f"De ellas, en la Región Metropolitana: {len(df_rcd_rm):,}")

TON_NACIONAL_2024 = df_rcd["cantidad_toneladas"].sum()
TON_RM_2024 = df_rcd_rm["cantidad_toneladas"].sum()
print(f"\nTonelaje RCD nacional 2024: {TON_NACIONAL_2024:,.0f} t")
print(f"Tonelaje RCD Región Metropolitana 2024: {TON_RM_2024:,.0f} t  ({TON_RM_2024/TON_NACIONAL_2024*100:.1f}% del nacional)")


Filas totales RETC/SINADER 2024 (todos los residuos, todo Chile): 39,395
Declaraciones de RCD (LER cap. 17) a nivel nacional: 4,463
De ellas, en la Región Metropolitana: 2,345

Tonelaje RCD nacional 2024: 1,808,162 t
Tonelaje RCD Región Metropolitana 2024: 956,012 t  (52.9% del nacional)


In [ ]:
# Tratamiento declarado por tonelaje en la RM
ton_tratamiento = df_rcd_rm.groupby("tratamiento_n1_name")["cantidad_toneladas"].sum().sort_values(ascending=False)
pct_tratamiento = ton_tratamiento / ton_tratamiento.sum() * 100
print("Tratamiento del RCD declarado en la RM, por tonelaje:")
display(pd.concat([ton_tratamiento.round(1), pct_tratamiento.round(1)], axis=1, keys=["toneladas", "%"]))


Tratamiento del RCD declarado en la RM, por tonelaje:
                            toneladas     %
tratamiento_n1_name                        
Eliminación                854,471.60 89.40
Valorización                95,548.20 10.00
Recepción y Almacenamiento   5,992.40  0.60


In [ ]:
# Regemac dentro del propio RETC — aparece como destinatario declarante
NOMBRE_REGEMAC_RETC = "REGENERADORA"  # razón social: "REGENERADORA DE MATERIALES DE CONSTRUCCION S A"

tonelaje_actor = df_rcd_rm.groupby("trazabilidad__razon_social")["cantidad_toneladas"].sum().sort_values(ascending=False)
ranking = tonelaje_actor.reset_index(drop=True)
fila_regemac = tonelaje_actor[tonelaje_actor.index.str.contains(NOMBRE_REGEMAC_RETC, case=False, na=False)]

TON_REGEMAC_RETC_2024 = float(fila_regemac.iloc[0])
rank_regemac = int(tonelaje_actor.index.get_loc(fila_regemac.index[0])) + 1
n_actores = len(tonelaje_actor)
SHARE_2024 = TON_REGEMAC_RETC_2024 / TON_RM_2024 * 100

print(f"Regemac declarado al RETC 2024: {TON_REGEMAC_RETC_2024:,.2f} t")
print(f"Ranking: #{rank_regemac} de {n_actores} destinatarios de la RM")
print(f"Participación de mercado 2024 = {TON_REGEMAC_RETC_2024:,.0f} / {TON_RM_2024:,.0f} = {SHARE_2024:.2f}%")

# Composición LER de lo que declara Regemac, y tratamiento asignado
reg_rows = df_rcd_rm[df_rcd_rm["trazabilidad__razon_social"].str.contains(NOMBRE_REGEMAC_RETC, case=False, na=False)]
print("\nComposición LER declarada por Regemac (t):")
display(reg_rows.groupby("LER")["cantidad_toneladas"].sum().sort_values(ascending=False))
print("\nTratamiento declarado por Regemac:")
display(reg_rows.groupby("tratamiento_n1_name")["cantidad_toneladas"].sum())


Regemac declarado al RETC 2024: 118,456.78 t
Ranking: #4 de 88 destinatarios de la RM
Participación de mercado 2024 = 118,457 / 956,012 = 12.39%

Composición LER declarada por Regemac (t):
LER
17 09 04 | Residuos mezclados de construcción y demolición distintos de los especificados en los códigos 17 09 01, 17 09 02 y 17 09 03   98,587.14
17 01 01 | Hormigón                                                                                                                      11,990.00
17 08 02 | Materiales de construcción a base de yeso distintos de los especificados en el código 17 08 01                                 6,212.85
17 05 04 | Tierra y piedras distintas de las especificadas en el código 17 05 03                                                          1,645.29
17 03 02 | Mezclas bituminosas distintas de las especificadas en el código 17 03 01                                                          10.00
17 01 07 | Mezclas de hormigón, ladrillos, tejas y materiales cerámicos,

In [ ]:
# Índice de Herfindahl-Hirschman (concentración de mercado), por tonelaje
participaciones_ton = tonelaje_actor / tonelaje_actor.sum()
IHH_TONELAJE = ((participaciones_ton * 100) ** 2).sum()
print(f"IHH por tonelaje recibido (RM, RETC 2024): {IHH_TONELAJE:,.0f}")
print("Referencia: <1.500 mercado no concentrado · 1.500-2.500 moderado · >2.500 alta concentración")


IHH por tonelaje recibido (RM, RETC 2024): 1,458
Referencia: <1.500 mercado no concentrado · 1.500-2.500 moderado · >2.500 alta concentración


In [ ]:
# Mercados "nicho" comprador de material valorizado en la RM (para contrastar con las ventas de Regemac, sección 5)
valorizado_rm = df_rcd_rm[df_rcd_rm["tratamiento_n1_name"] == "Valorización"].copy()

MERCADO_METALES_RM = valorizado_rm[valorizado_rm["LER_numero_subcapitulo"] == 4]["cantidad_toneladas"].sum()
MERCADO_MADERA_VIDRIO_PLASTICO_RM = valorizado_rm[valorizado_rm["LER_numero_subcapitulo"] == 2]["cantidad_toneladas"].sum()

print(f"Mercado comprador de metales valorizados, RM 2024: {MERCADO_METALES_RM:,.1f} t/año")
print(f"Mercado comprador madera/vidrio/plástico, RM 2024: {MERCADO_MADERA_VIDRIO_PLASTICO_RM:,.1f} t/año")


Mercado comprador de metales valorizados, RM 2024: 91,344.9 t/año
Mercado comprador madera/vidrio/plástico, RM 2024: 2,700.8 t/año


## 2. Proyección del mercado 2025-2026

El RETC solo está consolidado hasta 2024. No existe una proyección sectorial directa de generación de RCD, así que se usa como proxy el crecimiento de la inversión en construcción estimado por la CChC (Informe MACh, CDT, 20-dic-2025): **+2,2% en 2025** y **+4,8% en 2026**, asumiendo elasticidad 1:1 frente a la generación física de RCD.

*Candidato a análisis de sensibilidad — rango plausible de elasticidad: 0,5×–1,5× el crecimiento de inversión.*


In [ ]:
CRECIMIENTO_2025 = 0.022   # CChC, Informe MACh, dic-2025
CRECIMIENTO_2026 = 0.048   # CChC, Informe MACh, dic-2025

MERCADO_RM_2025 = TON_RM_2024 * (1 + CRECIMIENTO_2025)
MERCADO_RM_2026 = MERCADO_RM_2025 * (1 + CRECIMIENTO_2026)

print(f"Mercado RM 2024 (real, RETC):        {TON_RM_2024:>12,.0f} t")
print(f"Mercado RM 2025 (proyectado, +2,2%): {MERCADO_RM_2025:>12,.0f} t")
print(f"Mercado RM 2026 (proyectado, +4,8%): {MERCADO_RM_2026:>12,.0f} t")


Mercado RM 2024 (real, RETC):             956,012 t
Mercado RM 2025 (proyectado, +2,2%):      977,045 t
Mercado RM 2026 (proyectado, +4,8%):    1,023,943 t


## 3. Volumen gestionado por Regemac — guías de despacho

La hoja `Calculos` del archivo de guías es un pivote interno: OBRA × mes, con el conteo de retiros (contenedores llenos retirados) por obra y mes, más una columna `Volumen` con el tamaño de contenedor (m³) asignado a cada obra — disponible solo para una parte de las obras.

**Supuesto operativo**: la báscula no está operativa, pero se nos dijo que se consideraban llenos, así que cada contenedor retirado se considera 100% lleno.


In [ ]:
raw = pd.read_excel(PATH_GUIAS, sheet_name="Calculos", header=None)
years_row = raw.iloc[0]
months_row = raw.iloc[1]
data = raw.iloc[2:].reset_index(drop=True)

# Bloque A: columnas 1-68 = conteo de retiros por obra-mes, enero 2021 a agosto 2026
cols_bloqueA = list(range(1, 69))
labels_bloqueA = [f"{int(years_row[c])}-{int(months_row[c]):02d}" for c in cols_bloqueA]
bloqueA = data[cols_bloqueA].apply(pd.to_numeric, errors="coerce")
bloqueA.columns = labels_bloqueA

# Filtrar filas de resumen ("Vueltas", "# OBRAS", etc.) que no son una obra real
obra_num = pd.to_numeric(data[0], errors="coerce")
filas_validas = obra_num.notna()
print(f"Filas válidas (OBRA numérica): {filas_validas.sum()} de {len(data)}")

bloqueA = bloqueA[filas_validas].reset_index(drop=True)
volumen_obra = pd.to_numeric(data[141], errors="coerce")[filas_validas].reset_index(drop=True)

obras_con_volumen = volumen_obra.notna()
print(f"Obras con tamaño de contenedor (Volumen) conocido: {obras_con_volumen.sum()} de {len(volumen_obra)}")


Filas válidas (OBRA numérica): 1073 de 1107
Obras con tamaño de contenedor (Volumen) conocido: 501 de 1073


In [ ]:
# Densidad de retiros: m3 promedio ponderado por retiro, usando solo las obras con Volumen conocido
retiros_totales_conocidos = bloqueA[obras_con_volumen].sum().sum()
m3_totales_conocidos = (bloqueA[obras_con_volumen].sum(axis=1) * volumen_obra[obras_con_volumen]).sum()
M3_PROMEDIO_PONDERADO = m3_totales_conocidos / retiros_totales_conocidos
print(f"m³ promedio ponderado por retiro (obras con Volumen conocido): {M3_PROMEDIO_PONDERADO:.4f} m³/retiro")

def volumen_m3_bruto(anio):
    cols_anio = [c for c in bloqueA.columns if c.startswith(str(anio))]
    retiros_conocidos = bloqueA.loc[obras_con_volumen, cols_anio].sum().sum()
    m3_conocidos = (bloqueA.loc[obras_con_volumen, cols_anio].sum(axis=1) * volumen_obra[obras_con_volumen]).sum()
    retiros_desconocidos = bloqueA.loc[~obras_con_volumen, cols_anio].sum().sum()
    m3_desconocidos_est = retiros_desconocidos * M3_PROMEDIO_PONDERADO
    return dict(
        retiros=retiros_conocidos + retiros_desconocidos,
        m3=m3_conocidos + m3_desconocidos_est,
    )

RES_2024_BRUTO = volumen_m3_bruto(2024)
RES_2025_BRUTO = volumen_m3_bruto(2025)
print(f"\n2024 (bruto, vía pivote): {RES_2024_BRUTO['retiros']:,.0f} retiros / {RES_2024_BRUTO['m3']:,.0f} m³")
print(f"2025 (bruto, vía pivote): {RES_2025_BRUTO['retiros']:,.0f} retiros / {RES_2025_BRUTO['m3']:,.0f} m³")


m³ promedio ponderado por retiro (obras con Volumen conocido): 9.4853 m³/retiro

2024 (bruto, vía pivote): 24,468 retiros / 231,227 m³
2025 (bruto, vía pivote): 26,579 retiros / 251,902 m³


## 4. Validación cruzada con despachos granulares de 2026

Para 2026 sí existen los despachos con el detalle real de cada envío (`Reporte envíos [Mes].csv`), con el tamaño real de contenedor en `Tipo contenedor`. Se usa `Tipo de envío == 'normal'` (retiro de contenedor lleno; se excluyen `ida_contenedor` / `regreso_contenedor`, que son movimientos de contenedores vacíos).

Esto permite **comprobar si el pivote `Calculos` de la sección 3 subestima o no** los retiros reales, comparando el mismo período (enero-agosto) por dos vías independientes.


In [ ]:
filas = []
m3_directo_total = 0
retiros_directo_total = 0

for mes, path in PATHS_ENVIOS.items():
    df_mes = pd.read_csv(path, encoding="utf-8")
    df_mes.columns = [c.strip() for c in df_mes.columns]
    df_mes["Tipo contenedor"] = df_mes["Tipo contenedor"].astype(str).str.strip()
    normales = df_mes[df_mes["Tipo de envío"] == "normal"].copy()
    normales["m3"] = normales["Tipo contenedor"].str.extract(r"(\d+)").astype(float)
    n = len(normales)
    m3_mes = normales["m3"].sum()
    filas.append((mes, n, m3_mes))
    retiros_directo_total += n
    m3_directo_total += m3_mes

tabla_2026 = pd.DataFrame(filas, columns=["mes", "retiros", "m3"])
display(tabla_2026)
print(f"\nTOTAL directo enero-agosto 2026: {retiros_directo_total:,} retiros / {m3_directo_total:,.0f} m³")


       mes  retiros        m3
0    Enero     2033 20,300.00
1  Febrero     2095 21,064.00
2    Marzo     2689 29,518.00
3    Abril     2557 28,289.00
4     Mayo     2061 21,228.00
5    Junio     2388 24,283.00
6    Julio     2277 23,858.00
7   Agosto     2713 31,787.00

TOTAL directo enero-agosto 2026: 18,813 retiros / 200,327 m³


In [ ]:
# El mismo período (2026, ene-ago), pero calculado con el método del pivote interno (sección 3)
cols_2026_8m = [c for c in bloqueA.columns if c.startswith("2026")]
retiros_pivote_8m = bloqueA[cols_2026_8m].sum().sum()

FACTOR_CORRECCION = retiros_directo_total / retiros_pivote_8m
print(f"Retiros ene-ago 2026, vía pivote interno 'Calculos': {retiros_pivote_8m:,.0f}")
print(f"Retiros ene-ago 2026, contados directo en despachos: {retiros_directo_total:,.0f}")
print(f"\n>> El pivote interno SUBESTIMA los retiros reales en {(FACTOR_CORRECCION-1)*100:.1f}% <<")
print("Se aplica este factor de corrección a 2024 y 2025, que solo se pueden calcular vía el pivote.")


Retiros ene-ago 2026, vía pivote interno 'Calculos': 17,162
Retiros ene-ago 2026, contados directo en despachos: 18,813

>> El pivote interno SUBESTIMA los retiros reales en 9.6% <<
Se aplica este factor de corrección a 2024 y 2025, que solo se pueden calcular vía el pivote.


In [ ]:
M3_2024_CORREGIDO = RES_2024_BRUTO["m3"] * FACTOR_CORRECCION
M3_2025_CORREGIDO = RES_2025_BRUTO["m3"] * FACTOR_CORRECCION
M3_2026_PROYECTADO = m3_directo_total * 12 / 8   # anualización directa desde el dato granular real

print(f"m³ 2024 (corregido):        {M3_2024_CORREGIDO:>10,.0f}")
print(f"m³ 2025 (corregido):        {M3_2025_CORREGIDO:>10,.0f}")
print(f"m³ 2026 (proyectado x12/8): {M3_2026_PROYECTADO:>10,.0f}")


m³ 2024 (corregido):           253,471
m³ 2025 (corregido):           276,136
m³ 2026 (proyectado x12/8):    300,490


## 5. Conversión m3 -> toneladas: densidad calibrada

Regemac no tiene báscula operativa, así que no hay una densidad medida. Se calibra usando el único punto con tonelaje verificado de forma **independiente**: lo que Regemac mismo declaró al RETC en 2024 (sección 1).

*Candidato a análisis de sensibilidad — rango de la literatura para RCD mixto suelto no compactado: 0,40-0,55 t/m³.*


In [ ]:
DENSIDAD_CALIBRADA = TON_REGEMAC_RETC_2024 / M3_2024_CORREGIDO
print(f"Densidad calibrada = {TON_REGEMAC_RETC_2024:,.0f} t (RETC 2024) / {M3_2024_CORREGIDO:,.0f} m³ (guías, corregido) = {DENSIDAD_CALIBRADA:.4f} t/m³")

TON_REGEMAC_2024 = TON_REGEMAC_RETC_2024          # dato real, no requiere densidad
TON_REGEMAC_2025 = M3_2025_CORREGIDO * DENSIDAD_CALIBRADA
TON_REGEMAC_2026 = M3_2026_PROYECTADO * DENSIDAD_CALIBRADA

SHARE_2025 = TON_REGEMAC_2025 / MERCADO_RM_2025 * 100
SHARE_2026 = TON_REGEMAC_2026 / MERCADO_RM_2026 * 100

resumen_participacion = pd.DataFrame({
    "Mercado RM (t)": [TON_RM_2024, MERCADO_RM_2025, MERCADO_RM_2026],
    "Regemac (t)": [TON_REGEMAC_2024, TON_REGEMAC_2025, TON_REGEMAC_2026],
    "Participación (%)": [SHARE_2024, SHARE_2025, SHARE_2026],
}, index=["2024 (real)", "2025 (estimado)", "2026 (proyectado)"]).round(2)

display(resumen_participacion)


Densidad calibrada = 118,457 t (RETC 2024) / 253,471 m³ (guías, corregido) = 0.4673 t/m³
                   Mercado RM (t)  Regemac (t)  Participación (%)
2024 (real)            956,012.23   118,456.78              12.39
2025 (estimado)        977,044.50   129,048.83              13.21
2026 (proyectado)    1,023,942.64   140,430.79              13.71


## 6. Ventas de materiales valorizados

Registro de ventas (`Materiales_reciclados.xlsx`, hoja `Total`). Se excluyen las filas con `PRECIO < 0`: corresponden a pagos de Regemac a un tercero (RECUPAC) por **retirar** madera, es decir, un costo operacional, no un ingreso por venta.


In [ ]:
ventas = pd.read_excel(PATH_MATERIALES, sheet_name="Total")

costos_excluidos = ventas[(ventas["año"] == 2024) & (ventas["PRECIO"] < 0)]
print(f"Filas excluidas por PRECIO<0 (costo, no venta): {len(costos_excluidos)} filas, "
      f"{costos_excluidos['KG'].sum():,.0f} kg, ${costos_excluidos['PRECIO'].sum():,.0f} "
      f"(pago a RECUPAC por retiro de madera en 2024 — fuera del alcance de este punto de mercado)")

ventas_brutas = ventas[ventas["PRECIO"] > 0].copy()

def resumen_ventas(anio, hasta=None):
    sub = ventas_brutas[ventas_brutas["año"] == anio]
    if hasta is not None:
        sub = sub[sub["FECHA"] <= pd.Timestamp(hasta)]
    return sub["KG"].sum(), sub["PRECIO"].sum()

KG_2024, MONTO_2024 = resumen_ventas(2024)
KG_2025, MONTO_2025 = resumen_ventas(2025)
KG_2026_8M, MONTO_2026_8M = resumen_ventas(2026, hasta="2026-08-31")
KG_2026_PROY, MONTO_2026_PROY = KG_2026_8M * 12 / 8, MONTO_2026_8M * 12 / 8

print(f"\n2024: {KG_2024:>10,.0f} kg / ${MONTO_2024:>14,.0f}")
print(f"2025: {KG_2025:>10,.0f} kg / ${MONTO_2025:>14,.0f}")
print(f"2026 (ene-ago):    {KG_2026_8M:>10,.0f} kg / ${MONTO_2026_8M:>14,.0f}")
print(f"2026 (proyectado): {KG_2026_PROY:>10,.0f} kg / ${MONTO_2026_PROY:>14,.0f}")


Filas excluidas por PRECIO<0 (costo, no venta): 559 filas, 481,520 kg, $-8,185,840 (pago a RECUPAC por retiro de madera en 2024 — fuera del alcance de este punto de mercado)

2024:    885,408 kg / $   103,220,672
2025:    859,302 kg / $   114,179,800
2026 (ene-ago):       492,020 kg / $    82,293,440
2026 (proyectado):    738,030 kg / $   123,440,160


In [ ]:
# Composición por material, año 2025 (año completo más representativo)
comp_2025 = ventas_brutas[ventas_brutas["año"] == 2025].groupby("MATERIAL").agg(KG=("KG", "sum"), MONTO=("PRECIO", "sum"))
comp_2025["% KG"] = (comp_2025["KG"] / comp_2025["KG"].sum() * 100).round(2)
comp_2025["% MONTO"] = (comp_2025["MONTO"] / comp_2025["MONTO"].sum() * 100).round(2)
display(comp_2025.sort_values("MONTO", ascending=False))


                   KG          MONTO  % KG  % MONTO
MATERIAL                                           
FIERRO     552,640.00 100,513,370.00 64.31    88.03
MADERA     257,730.00   5,016,080.00 29.99     4.39
ALUMINIO     2,720.00   4,093,490.00  0.32     3.59
CARTON      30,841.00   2,993,310.00  3.59     2.62
NEUMATICOS   5,300.00   1,060,000.00  0.62     0.93
PLASTICO    10,071.00     503,550.00  1.17     0.44


In [ ]:
# Participación de Regemac en los mercados "nicho" (sección 1) — solo fierro y madera, 2025
FIERRO_2025_T = comp_2025.loc["FIERRO", "KG"] / 1000
MADERA_2025_T = comp_2025.loc["MADERA", "KG"] / 1000

print(f"Fierro vendido 2025: {FIERRO_2025_T:,.2f} t -> "
      f"{FIERRO_2025_T / MERCADO_METALES_RM * 100:.2f}% del mercado comprador de metales de la RM "
      f"({MERCADO_METALES_RM:,.0f} t/año)")
print(f"Madera vendida 2025: {MADERA_2025_T:,.2f} t -> "
      f"{MADERA_2025_T / MERCADO_MADERA_VIDRIO_PLASTICO_RM * 100:.2f}% del nicho madera/vidrio/plástico de la RM "
      f"({MERCADO_MADERA_VIDRIO_PLASTICO_RM:,.0f} t/año)")

print(f"\nTasa de recuperación másica 2025 = ventas / volumen total gestionado "
      f"= {MADERA_2025_T + FIERRO_2025_T + comp_2025['KG'].sum()/1000 - MADERA_2025_T - FIERRO_2025_T:,.0f} t "
      f"/ {TON_REGEMAC_2025:,.0f} t = {comp_2025['KG'].sum()/1000 / TON_REGEMAC_2025 * 100:.2f}%")


Fierro vendido 2025: 552.64 t -> 0.61% del mercado comprador de metales de la RM (91,345 t/año)
Madera vendida 2025: 257.73 t -> 9.54% del nicho madera/vidrio/plástico de la RM (2,701 t/año)

Tasa de recuperación másica 2025 = ventas / volumen total gestionado = 859 t / 129,049 t = 0.67%


## 7. Tabla síntesis


In [ ]:
tabla_final = pd.DataFrame({
    "2024": [TON_RM_2024, TON_REGEMAC_2024, SHARE_2024, KG_2024, MONTO_2024],
    "2025": [MERCADO_RM_2025, TON_REGEMAC_2025, SHARE_2025, KG_2025, MONTO_2025],
    "2026 (proy.)": [MERCADO_RM_2026, TON_REGEMAC_2026, SHARE_2026, KG_2026_PROY, MONTO_2026_PROY],
}, index=[
    "Mercado RCD declarado, RM (t)",
    "Regemac — volumen gestionado (t)",
    "Participación de mercado Regemac (%)",
    "Ventas materiales valorizados (kg)",
    "Ventas materiales valorizados (CLP)",
])
display(tabla_final.round(2))


                                               2024           2025   2026 (proy.)
Mercado RCD declarado, RM (t)            956,012.23     977,044.50   1,023,942.64
Regemac — volumen gestionado (t)         118,456.78     129,048.83     140,430.79
Participación de mercado Regemac (%)          12.39          13.21          13.71
Ventas materiales valorizados (kg)       885,408.00     859,302.00     738,030.00
Ventas materiales valorizados (CLP)  103,220,672.00 114,179,800.00 123,440,160.00


## 8. Tabla de supuestos

| Supuesto | Valor asumido | Razonamiento / fuente | Rango — sensibilidad |
|---|---|---|---|
| Llenado de contenedor | 100% | Instrucción operativa Regemac: báscula no operativa | 70–100%; sí, prioritaria |
| Crecimiento mercado RCD 2025-26 | +2,2% / +4,8% | Proxy: CChC, Informe MACh (dic-2025); elasticidad 1:1 | Elasticidad 0,5×–1,5×; sí |
| Densidad RCD mixto | ~0,467 t/m³ (calculada arriba) | Calibrada contra RETC 2024 | 0,40–0,55 t/m³; sí |
| Corrección de conteo de retiros | ~+9,6% (calculada arriba) | Validación cruzada ene-ago 2026 | 0–15%; sí |

Nota adicional (no estaba en el docx, se agregó tras revisión): el RETC puede subdeclarar de forma distinta a Regemac que al resto de los actores de la RM. Si Regemac declara de forma más completa que sus competidores, la participación de mercado real de Regemac sería **mayor** a la calculada aquí (el denominador estaría subestimado); si fuera al revés, sería menor. No hay información disponible hoy para saber en qué dirección pesa más — es un sesgo de dirección desconocida, distinto de los otros supuestos, que no admite un rango numérico de sensibilidad todavía.
